# 03. 순위 표현은 정말 더 견고한가? — 깊이 변화 · 입력 ablation

**등급**: 🟢 Mac 로컬 (MPS/CPU) · **예상 시간**: 약 20분 (대부분 V2 추론) · **선행**: 노트북 01 · **다운로드**: 없음

**질문**
1. Geneformer 설명은 "순위 기반 표현은 기술적 아티팩트에 더 견고하다"고 한다. **시퀀싱 깊이가 달라지면**(참조는 깊게, 질의는 얕게) 임베딩 기반 세포 유형 분류가 PCA보다 덜 무너지는가?
2. 임베딩은 토큰열의 **무엇**에 의존하는가? (a) 검출 여부 집합, (b) 카운트에서 오는 순위, (c) 앞쪽 상위 유전자만, (d) 순서 자체.

**설계**
- 하위표본: 기증자당 ≈300개, 총 약 4,000세포 (노트북 01의 10.4k 하위표본에서 시드 0으로 뽑음). 속도 때문이며 희귀 세포 유형의 지표는 흔들린다 → 기증자 단위 부트스트랩 구간으로 표시.
- 깊이 변화: 원시 카운트를 **이항 솎아내기**(각 카운트를 확률 p로 유지; p=0.5, 0.25)해 얕은 시퀀싱을 흉내낸다. n_counts도 솎아낸 행렬에서 다시 계산한다. 분류기는 **솎아내지 않은 다른 12명**의 임베딩으로 학습하고, held-out 기증자의 **솎아낸 세포**를 예측한다 (LODO).
- 비교 대상: PCA(HVG) (참조 세포에서 적합한 HVG·표준화·PCA에 솎아낸 세포를 **투영**), Geneformer V1-10M, V2-104M. scVI는 새 데이터 변환 절차가 달라 제외했다.
- 이 솎아내기는 **이항 다운샘플링 시뮬레이션**이다. 실제 얕은 시퀀싱/다른 플랫폼의 차이(유전자별 검출 편향, 앰비언트 RNA 등)를 대표하지 않는다.

In [ ]:
import hashlib
import io
import os
import pickle
import time
import warnings
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold, cross_val_predict

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "SETUP.md").exists():
            return q
    return None


ROOT = find_root()
CACHE = (ROOT / ".cache") if ROOT is not None else Path("cache")
os.environ.setdefault("HF_HOME", str(CACHE / "hf"))
SC = CACHE / "singlecell"
for need in ["pbmc_sub.h5ad", "geneformer_embeddings.npz"]:
    assert (SC / need).exists(), f"{SC / need} 가 없다 — 노트북 01을 먼저 실행하라"

import torch
from huggingface_hub import hf_hub_download
from transformers import BertConfig, BertModel
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)
print("device:", DEVICE)

## 0. 노트북 01의 구성요소 (독립 실행을 위해 복사)
허용 목록 Unpickler, `RankTokenizer`, 임베딩 추출. 동작은 노트북 01과 같다 (설명은 01 참고).

In [ ]:
REV = "1f7fbae4e469a5f4f1af8c111a529cfe1b3829f5"
REPO = "ctheodoris/Geneformer"
FILES = {
    "v1": dict(model="Geneformer-V1-10M", tok="geneformer/gene_dictionaries_30m/token_dictionary_gc30M.pkl",
               med="geneformer/gene_dictionaries_30m/gene_median_dictionary_gc30M.pkl",
               map="geneformer/gene_dictionaries_30m/ensembl_mapping_dict_gc30M.pkl", max_len=2048, special=False),
    "v2": dict(model="Geneformer-V2-104M", tok="geneformer/token_dictionary_gc104M.pkl",
               med="geneformer/gene_median_dictionary_gc104M.pkl",
               map="geneformer/ensembl_mapping_dict_gc104M.pkl", max_len=4096, special=True),
}


def fetch(rel):
    return hf_hub_download(REPO, rel, revision=REV)


class SafeUnpickler(pickle.Unpickler):
    ALLOWED = {("numpy", "dtype"), ("numpy.core.multiarray", "scalar"), ("numpy._core.multiarray", "scalar")}

    def find_class(self, module, name):
        if (module, name) in self.ALLOWED:
            return super().find_class(module, name)
        raise pickle.UnpicklingError(f"blocked global: {module}.{name}")


def load_pkl(path):
    with open(path, "rb") as f:
        return SafeUnpickler(f).load()


class RankTokenizer:
    def __init__(self, version):
        cfg = FILES[version]
        self.tok = load_pkl(fetch(cfg["tok"]))
        self.med = load_pkl(fetch(cfg["med"]))
        mp = load_pkl(fetch(cfg["map"]))
        self.map = {k: v for k, v in mp.items() if v in self.tok}
        self.genes = [g for g in self.tok if g.startswith("ENS")]
        self.gidx = {g: i for i, g in enumerate(self.genes)}
        self.token_ids = np.array([self.tok[g] for g in self.genes])
        self.median = np.array([self.med[g] for g in self.genes])
        self.max_len, self.special = cfg["max_len"], cfg["special"]

    def collapse(self, counts, var_ids):
        mapped = pd.Series(np.asarray(var_ids)).str.upper().map(self.map)
        keep = np.where(mapped.notna().values)[0]
        cols = [self.gidx[mapped.iloc[i]] for i in keep]
        G = sp.csr_matrix((np.ones(len(keep)), (np.arange(len(keep)), cols)), shape=(len(keep), len(self.genes)))
        return (counts[:, keep] @ G).tocsr()

    def normalize_collapsed(self, Xc, n_counts, target_sum=10_000):
        return (sp.diags(target_sum / n_counts) @ Xc @ sp.diags(1.0 / self.median)).tocsr()

    def to_tokens(self, Xn, max_genes=None):
        L = self.max_len - (2 if self.special else 0)
        if max_genes is not None:
            L = min(L, max_genes)
        out = []
        for i in range(Xn.shape[0]):
            s, e = Xn.indptr[i], Xn.indptr[i + 1]
            v, ix = Xn.data[s:e], Xn.indices[s:e]
            t = self.token_ids[ix[np.argsort(-v, kind="stable")]][:L]
            if self.special:
                t = np.concatenate([[self.tok["<cls>"]], t, [self.tok["<eos>"]]])
            out.append(t.astype(np.int64))
        return out


@torch.no_grad()
def embed(model, token_lists, special, cls_id=None, eos_id=None, batch=32, layer=-2):
    n, H = len(token_lists), model.config.hidden_size
    order = np.argsort([len(t) for t in token_lists])
    mean_ = np.zeros((n, H), np.float32)
    cls_ = np.zeros((n, H), np.float32)
    for b in range(0, n, batch):
        bi = order[b:b + batch]
        ml = max(len(token_lists[i]) for i in bi)
        ids_ = torch.zeros(len(bi), ml, dtype=torch.long)
        for j, i in enumerate(bi):
            ids_[j, :len(token_lists[i])] = torch.from_numpy(token_lists[i])
        ids_ = ids_.to(DEVICE)
        att = ids_ != 0
        h = model(input_ids=ids_, attention_mask=att.long(), output_hidden_states=True).hidden_states[layer]
        gm = att.clone()
        if special:
            gm &= (ids_ != cls_id) & (ids_ != eos_id)
        m = gm.unsqueeze(-1).float()
        mean_[bi] = ((h * m).sum(1) / m.sum(1)).float().cpu().numpy()
        cls_[bi] = h[:, 0].float().cpu().numpy()
    return mean_, cls_


def load_model(version):
    path = Path(fetch(FILES[version]["model"] + "/config.json")).parent
    fetch(FILES[version]["model"] + "/model.safetensors")
    return BertModel.from_pretrained(path, add_pooling_layer=False).to(DEVICE).eval()


# 평가 함수 (노트북 02와 동일)
def prep(Z, center=True):
    Z = np.asarray(Z, dtype=np.float64)
    if center:
        Z = Z - Z.mean(0)
    return Z / np.linalg.norm(Z, axis=1, keepdims=True)


def knn_vote(Ztr, ytr, Zte, n_classes, k=15):
    S = Zte @ Ztr.T
    idx = np.argpartition(-S, k, axis=1)[:, :k]
    votes = np.zeros((len(Zte), n_classes))
    rows = np.arange(len(Zte))
    for j in range(k):
        np.add.at(votes, (rows, ytr[idx[:, j]]), 1.0 + 1e-3 * S[rows, idx[:, j]])
    return votes.argmax(1)


def macro_f1_from_cm(cm):
    tp = np.diagonal(cm, axis1=-2, axis2=-1)
    fp, fn = cm.sum(-2) - tp, cm.sum(-1) - tp
    den = 2 * tp + fp + fn
    f1 = np.where(den > 0, 2 * tp / np.where(den > 0, den, 1), np.nan)
    return np.nanmean(f1, axis=-1)


def ci(x):
    return np.percentile(x[~np.isnan(x)], [2.5, 97.5])

## 1. 하위표본과 참조 임베딩
노트북 01이 저장한 10.4k 세포 중 기증자당 ≈300개를 뽑는다. 참조(솎아내지 않은) 임베딩은 01에서 저장한 것을 그대로 쓴다.

In [ ]:
full = ad.read_h5ad(SC / "pbmc_sub.h5ad")
E = dict(np.load(SC / "geneformer_embeddings.npz"))
obs_full = full.obs
var_ids = np.array(full.var_names)
rng = np.random.default_rng(SEED)
sel = []
for d, g in obs_full.groupby("donor", observed=True).indices.items():
    sel += list(rng.choice(g, size=min(300, len(g)), replace=False))
sel = np.sort(np.array(sel))
obs = obs_full.iloc[sel]
counts0 = full.X[sel].tocsr() if sp.issparse(full.X) else sp.csr_matrix(full.X[sel])
counts0 = counts0.astype(np.float64)
labels, donors = obs.label.values, obs.donor.values
class_names, donor_names = sorted(obs.label.unique()), sorted(obs.donor.unique())
y = np.array([class_names.index(l) for l in labels])
print("하위표본:", counts0.shape, "| 세포 유형 분포:", dict(pd.Series(labels).value_counts()))

## 2. 이항 솎아내기로 얕은 시퀀싱 만들기
각 카운트 $c$를 $\mathrm{Binomial}(c, p)$로 바꾼다. 세포당 총 카운트와 검출 유전자 수가 어떻게 변하는지 먼저 본다.

In [ ]:
def thin(counts, p, seed):
    out = counts.copy().tocsr()
    out.data = np.random.default_rng(seed).binomial(out.data.astype(np.int64), p).astype(np.float64)
    out.eliminate_zeros()
    return out


P_LIST = [0.5, 0.25]
thinned = {1.0: counts0}
for p in P_LIST:
    thinned[p] = thin(counts0, p, SEED + int(p * 100))
rows = []
for p, X in thinned.items():
    nc = np.asarray(X.sum(1)).ravel()
    rows.append({"p": p, "세포당 총 카운트 중앙값": np.median(nc), "최소": nc.min(), "검출 유전자 중앙값": np.median(np.diff(X.indptr)),
                 "카운트=1 항목 비율": float(np.mean(X.data == 1))})
print(pd.DataFrame(rows).set_index("p").round(2).to_string())
assert all(np.asarray(X.sum(1)).ravel().min() > 0 for X in thinned.values()), "빈 세포 발생"

## 3. PCA(HVG) baseline — 참조에서 적합, 솎아낸 세포는 투영
참조(p=1.0) 세포로 HVG 2,000개 · 로그 정규화의 유전자별 평균/표준편차 · PCA 50을 적합하고, 솎아낸 세포에는 **같은 변환을 적용**한다 (재적합하지 않음). 이것이 "깊은 참조에 얕은 질의를 올리는" 상황이다.

In [ ]:
a_ref = ad.AnnData(X=counts0.astype(np.float32), var=pd.DataFrame(index=var_ids))
sc.pp.filter_genes(a_ref, min_cells=3)
sc.pp.highly_variable_genes(a_ref, n_top_genes=2000, flavor="seurat_v3")
hv = np.where(a_ref.var.highly_variable.values)[0]
gene_keep = a_ref.var_names.values        # filter_genes 이후 남은 유전자
col_idx = pd.Index(var_ids).get_indexer(gene_keep)[hv]


def lognorm(X):
    X = sp.diags(1e4 / np.asarray(X.sum(1)).ravel()) @ X
    X = X.tocsr()
    X.data = np.log1p(X.data)
    return X


ref_ln = lognorm(counts0)[:, col_idx].toarray()
mu, sd = ref_ln.mean(0), ref_ln.std(0, ddof=1)
sd[sd == 0] = 1
pca = PCA(n_components=50, random_state=SEED).fit(np.clip((ref_ln - mu) / sd, None, 10))


def pca_project(X):
    return pca.transform(np.clip((lognorm(X)[:, col_idx].toarray() - mu) / sd, None, 10))


Z = {"PCA(HVG,50)": {p: pca_project(X) for p, X in thinned.items()}}
print("PCA 투영 완료:", {p: v.shape for p, v in Z["PCA(HVG,50)"].items()})

## 4. Geneformer 임베딩 — p=1.0은 01의 저장값, 솎아낸 입력은 새로 계산

In [ ]:
t_v1, t_v2 = RankTokenizer("v1"), RankTokenizer("v2")
sel_in_full = sel          # 01 하위표본 안의 위치
Z["Geneformer V1-10M"] = {1.0: E["v1_mean_L-2"][sel_in_full]}
Z["Geneformer V2-104M"] = {1.0: E["v2_cls_L-2"][sel_in_full]}


def tokenize(t, X):
    Xc = t.collapse(X, var_ids)
    return t.to_tokens(t.normalize_collapsed(Xc, np.asarray(X.sum(1)).ravel())), Xc


# 재현 점검: 01의 저장 임베딩을 이 노트북의 토큰화로 다시 만들어 일치하는지 (V1, 30세포)
m1 = load_model("v1")
tk0, _ = tokenize(t_v1, counts0)
chk, _ = embed(m1, tk0[:30], False)
print("V1 재계산 vs 01 저장 임베딩 최대 절대차: %.2e" % np.abs(chk - Z["Geneformer V1-10M"][1.0][:30]).max())

t0 = time.time()
for p in P_LIST:
    tk, _ = tokenize(t_v1, thinned[p])
    Z["Geneformer V1-10M"][p] = embed(m1, tk, False)[0]
print("V1 솎아내기 임베딩: %.0f초" % (time.time() - t0))
del m1

m2 = load_model("v2")
cid, eid = t_v2.tok["<cls>"], t_v2.tok["<eos>"]
t0 = time.time()
tk0v2, _ = tokenize(t_v2, counts0)
chk2 = embed(m2, tk0v2[:30], True, cid, eid)[1]
print("V2 재계산 vs 01 저장(cls) 최대 절대차: %.2e" % np.abs(chk2 - Z["Geneformer V2-104M"][1.0][:30]).max())
for p in P_LIST:
    tk, _ = tokenize(t_v2, thinned[p])
    Z["Geneformer V2-104M"][p] = embed(m2, tk, True, cid, eid)[1]
print("V2 솎아내기 임베딩: %.0f초" % (time.time() - t0))
del m2
if DEVICE.type == "mps":
    torch.mps.empty_cache()

## 5. 깊이 변화 평가
- **분류**: 참조(p=1) 임베딩으로 학습한 kNN(k=15)이 held-out 기증자의 솎아낸 세포를 분류한 macro-F1과 정확도 (LODO). `p=1.0`은 학습·평가 모두 솎아내지 않은 경우(상한 기준).
- **자기 유사도**: 같은 세포의 참조 임베딩과 솎아낸 임베딩의 코사인 유사도 (참조 평균으로 중심화). 높다고 분류가 보존된다는 뜻은 아니다 — 모든 세포가 비슷해 보이는 임베딩에서도 높을 수 있다 (anisotropy).

In [ ]:
B = 1000
boot = np.random.default_rng(SEED).integers(0, len(donor_names), size=(B, len(donor_names)))
C = len(class_names)


def lodo_shift(Zref, Ztest):
    """참조 임베딩으로 학습, (솎아낸) 테스트 임베딩 예측. 중심화는 참조 평균으로 통일한다."""
    mu_ = Zref.mean(0)
    R_, T_ = prep(Zref - mu_, False), prep(Ztest - mu_, False)
    cms = np.zeros((len(donor_names), C, C))
    for i, d in enumerate(donor_names):
        te = donors == d
        pred = knn_vote(R_[~te], y[~te], T_[te], C)
        np.add.at(cms[i], (y[te], pred), 1)
    return cms


res, cms_store = [], {}
for name, byp in Z.items():
    for p in [1.0] + P_LIST:
        cms = lodo_shift(byp[1.0], byp[p])
        cms_store[(name, p)] = cms
        f1 = macro_f1_from_cm(cms.sum(0))
        fb = macro_f1_from_cm(cms[boot].sum(1))
        acc = np.trace(cms.sum(0)) / cms.sum()
        mu_ = byp[1.0].mean(0)
        a_, b_ = prep(byp[1.0] - mu_, False), prep(byp[p] - mu_, False)
        res.append(dict(방법=name, p=p, macroF1=f1, lo=ci(fb)[0], hi=ci(fb)[1], 정확도=acc, 자기유사도=float((a_ * b_).sum(1).mean())))
R = pd.DataFrame(res)
R["macro-F1 [95% CI]"] = R.apply(lambda r: f"{r.macroF1:.3f} [{r.lo:.3f}, {r.hi:.3f}]", axis=1)
print(R.pivot(index="방법", columns="p", values="macro-F1 [95% CI]").to_string())
print()
print(R.pivot(index="방법", columns="p", values="정확도").round(3).to_string())
print()
print("자기 유사도 (참조 평균 중심화 후 코사인):")
print(R.pivot(index="방법", columns="p", values="자기유사도").round(3).to_string())

# 짝지은 하락폭: p=1 대비 macro-F1 감소 (기증자 부트스트랩)
rows = []
for name in Z:
    for p in P_LIST:
        d = macro_f1_from_cm(cms_store[(name, p)][boot].sum(1)) - macro_f1_from_cm(cms_store[(name, 1.0)][boot].sum(1))
        dd = macro_f1_from_cm(cms_store[(name, p)].sum(0)) - macro_f1_from_cm(cms_store[(name, 1.0)].sum(0))
        rows.append({"방법": name, "p": p, "Δ macro-F1 (p vs 1.0)": f"{dd:+.3f} [{ci(d)[0]:+.3f}, {ci(d)[1]:+.3f}]"})
print("\n솎아내기로 인한 macro-F1 변화:")
print(pd.DataFrame(rows).pivot(index="방법", columns="p", values="Δ macro-F1 (p vs 1.0)").to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 3.6))
for name in Z:
    s = R[R.방법 == name].sort_values("p", ascending=False)
    ax.errorbar(s.p, s.macroF1, yerr=[s.macroF1 - s.lo, s.hi - s.macroF1], marker="o", capsize=3, label=name)
ax.set_xlabel("유지 확률 p (1.0 = 원본, 작을수록 얕은 시퀀싱)")
ax.set_ylabel("LODO macro-F1 (테스트만 솎아냄)")
ax.invert_xaxis()
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 6. 임베딩이 깊이(총 카운트)를 담고 있는가
임베딩에서 `log(총 카운트)`를 릿지 회귀로 5-fold 교차검증 예측한 $R^2$. 값이 크면 임베딩 축 일부가 기술적 깊이를 인코딩한다는 뜻이다 (깊이가 세포 유형과 상관되어 있으면 생물학 신호도 섞여 있을 수 있다).

In [ ]:
logn = np.log(np.asarray(thinned[1.0].sum(1)).ravel())
rows = []
for name, byp in Z.items():
    Xp = prep(byp[1.0] - byp[1.0].mean(0), False)
    pred = cross_val_predict(RidgeCV(alphas=np.logspace(-3, 3, 13)), Xp, logn, cv=KFold(5, shuffle=True, random_state=SEED))
    r2 = 1 - ((logn - pred) ** 2).sum() / ((logn - logn.mean()) ** 2).sum()
    rows.append({"임베딩": name, "log(총 카운트) 예측 R²": r2})
print(pd.DataFrame(rows).set_index("임베딩").round(3).to_string())
print("세포 유형만으로 설명되는 log(총 카운트) 분산 비율 (η²): %.3f" % (1 - sum(((logn[labels == l] - logn[labels == l].mean()) ** 2).sum() for l in class_names) / ((logn - logn.mean()) ** 2).sum()))

## 7. 입력 ablation — 임베딩은 토큰열의 무엇에 의존하는가
같은 하위표본에서 Geneformer V1 입력을 네 가지로 바꿔 **새 임베딩을 계산**하고, 각각 학습·평가(LODO)를 모두 그 입력으로 한다.
- **전체 순위**: 원래 방식.
- **검출 집합만(count-blind)**: 카운트를 모두 1로 바꿔 검출된 유전자를 코퍼스 중앙값의 역순으로만 정렬. 세포마다 **검출된 유전자 집합**만 다르고 카운트 정보는 없다. 이 입력도 모델이 보던 형식(중앙값 정렬된 순위열)이라 분포 밖이 아니다.
- **상위 256개만**: 순위 앞쪽 256개 유전자로 절단.
- **순서 무작위화**: 같은 유전자 집합을 세포마다 무작위로 섞음. 사전학습에서 본 적 없는 입력이라 성능 하락이 곧 "순서에 정보가 있다"는 증거는 아니다 (분포 밖).
V2는 시간 때문에 "검출 집합만"만 수행한다.

In [ ]:
def tokens_variant(t, X, kind, seed=SEED):
    Xc = t.collapse(X, var_ids)
    if kind == "full":
        return t.to_tokens(t.normalize_collapsed(Xc, np.asarray(X.sum(1)).ravel()))
    if kind == "top256":
        return t.to_tokens(t.normalize_collapsed(Xc, np.asarray(X.sum(1)).ravel()), max_genes=256)
    if kind == "detect_only":
        Xb = Xc.copy()
        Xb.data[:] = 1.0
        return t.to_tokens(t.normalize_collapsed(Xb, np.ones(Xb.shape[0]) * 1e4))
    if kind == "shuffle":
        r = np.random.default_rng(seed)
        return [r.permutation(x) for x in t.to_tokens(t.normalize_collapsed(Xc, np.asarray(X.sum(1)).ravel()))]


def lodo_same(Zx):
    Zn = prep(Zx)
    cms = np.zeros((len(donor_names), C, C))
    for i, d in enumerate(donor_names):
        te = donors == d
        pred = knn_vote(Zn[~te], y[~te], Zn[te], C)
        np.add.at(cms[i], (y[te], pred), 1)
    return cms


abl = {}
m1 = load_model("v1")
t0 = time.time()
for kind in ["full", "detect_only", "top256", "shuffle"]:
    tk = tokens_variant(t_v1, counts0, kind)
    L = np.array([len(x) for x in tk])
    abl[("V1", kind)] = (embed(m1, tk, False)[0], L.mean())
print("V1 ablation 임베딩: %.0f초" % (time.time() - t0))
del m1
m2 = load_model("v2")
t0 = time.time()
for kind in ["detect_only"]:
    tk = tokens_variant(t_v2, counts0, kind)
    abl[("V2", kind)] = (embed(m2, tk, True, cid, eid)[1], np.mean([len(x) for x in tk]))
abl[("V2", "full")] = (Z["Geneformer V2-104M"][1.0], np.mean([len(x) for x in tk0v2]))
print("V2 ablation 임베딩: %.0f초" % (time.time() - t0))
del m2
if DEVICE.type == "mps":
    torch.mps.empty_cache()

cms_abl = {k: lodo_same(Zx) for k, (Zx, L) in abl.items()}
rows = []
for (mdl, kind), (Zx, L) in abl.items():
    cms = cms_abl[(mdl, kind)]
    f1 = macro_f1_from_cm(cms.sum(0))
    fb = macro_f1_from_cm(cms[boot].sum(1))
    if kind == "full":
        delta = "—"
    else:
        d_ = fb - macro_f1_from_cm(cms_abl[(mdl, "full")][boot].sum(1))      # 같은 재표본으로 짝지은 차이
        delta = f"{f1 - macro_f1_from_cm(cms_abl[(mdl, 'full')].sum(0)):+.3f} [{ci(d_)[0]:+.3f}, {ci(d_)[1]:+.3f}]"
    rows.append({"모델": mdl, "입력": kind, "평균 토큰 수": round(L), "macro-F1 [95% CI]": f"{f1:.3f} [{ci(fb)[0]:.3f}, {ci(fb)[1]:.3f}]",
                 "Δ vs 전체 순위 [95% CI]": delta, "정확도": round(np.trace(cms.sum(0)) / cms.sum(), 3)})
print(pd.DataFrame(rows).set_index(["모델", "입력"]).to_string())

## 8. 해석과 한계
아래는 이 실행(seed 0, 13명×300세포 = 3,900세포)에서 읽은 내용이다. **표를 직접 확인하라.**

**깊이 변화**
- 총 카운트 중앙값을 1,484 → 741 (p=0.5) → 372 (p=0.25)로 줄이면 카운트=1 항목의 비율이 0.77 → 0.83 → 0.88로 늘고 검출 유전자 수 중앙값은 894 → 513 → 286으로 준다.
- 참조(깊은) 임베딩으로 학습하고 얕은 세포를 분류했을 때 macro-F1: **PCA 0.856 → 0.838 → 0.790**, V1 0.509 → 0.356 → 0.191, V2 0.802 → 0.741 → 0.527. p=1 대비 하락폭은 PCA −0.019 [−0.035, −0.002] / −0.066 [−0.096, −0.040], V1 −0.153 / −0.318, V2 −0.061 [−0.100, −0.033] / −0.275. **"순위 기반 표현이 깊이 차이에 더 견고하다"는 설명은 이 시뮬레이션에서 지지되지 않았다.** 로그 정규화 후 참조에서 적합한 PCA 투영이 훨씬 덜 무너졌다.
- 같은 세포의 참조 임베딩과 솎아낸 임베딩의 코사인 유사도(자기 유사도)는 PCA 0.92/0.78, V1 0.52/0.22, V2 0.67/0.37이다. 순위는 "덜 바뀌는" 것이 아니라 **검출 집합이 바뀌면 토큰열이 크게 바뀐다**: 얕은 시퀀싱에서는 낮은 발현 유전자가 사라지고 남은 유전자의 순위가 재배열된다.
- V2가 모든 조건에서 V1보다 높다. 그러나 두 모델은 크기(104M 대 10M), 사전학습 코퍼스(약 1억 대 3천만 세포), 입력 길이, 어휘, `<cls>` 사용이 함께 다르므로 **어느 요인 때문인지 이 실험으로는 분리할 수 없다.**

**깊이가 임베딩에 얼마나 들어 있는가**
- 임베딩에서 `log(총 카운트)`를 교차검증 릿지로 예측한 R²: PCA 0.63, V1 0.90, V2 0.93. 세포 유형이 설명하는 `log(총 카운트)` 분산은 5.7%뿐이므로 이 R²의 대부분은 **세포 유형이 아니라 기술적 깊이**와 연결된 방향으로 보인다(깊이와 다른 기술 요인이 섞여 있을 수 있음). 이것이 얕은 질의에서 임베딩이 크게 움직이는 한 이유라는 **가설**이다(원인은 직접 시험하지 않음).

**입력 ablation**
- V1: 전체 순위 0.509 [0.442, 0.542] / **검출 집합만(카운트 정보 제거) 0.454 [0.396, 0.484]** / 상위 256개만 0.453 / 순서 무작위화 0.300. V2: 전체 0.802 / 검출 집합만 0.733 [0.664, 0.776].
- 짝지은 차이(전체 순위 대비): 검출 집합만 → **V1 −0.056 [−0.076, −0.033], V2 −0.069 [−0.090, −0.043]**, 상위 256개만(V1) −0.056 [−0.086, −0.024], 순서 무작위화(V1) −0.209 [−0.239, −0.182]. 카운트 정보를 전부 지워도 0.5/0.8 수준의 성능에서 6~7점(상대적으로 약 9~11%)만 줄어든다 — **카운트에서 오는 순위 정보의 기여는 0이 아니지만 작다**. 노트북 02의 무작위 초기화 결과와 함께 보면 **이 데이터에서 임베딩의 세포 유형 신호는 상당 부분 \"어떤 유전자가 검출되었는가\"에서 온다.** 얕은 데이터(카운트=1이 77%)라는 조건과 결합된 결과이며 깊은 데이터에도 같다고 말할 수 없다.
- 순서를 무작위화하면 크게 떨어지지만 이는 사전학습에서 본 적 없는 **분포 밖 입력**이므로 "순서에 생물학 정보가 있다"는 증거로 해석하면 안 된다.

**한계**
- 이항 솎아내기는 **시뮬레이션**이다. 실제 얕은 시퀀싱·다른 플랫폼은 유전자별 검출 편향, 앰비언트 RNA 등이 다르다. 솎아낸 p=0.25 세포는 총 카운트가 최소 143까지 내려가 사전학습 분포와 멀 수 있다.
- PCA는 "참조에서 적합 후 투영"이라는 한 가지 방식만 봤다. 얕은 세포를 함께 섞어 학습하는 깊이 증강, Geneformer 임베딩 위의 지도 분류기(probe) 등은 시험하지 않았다 → 결론은 **zero-shot 임베딩 + 깊은 참조 kNN**에 한정된다.
- 하위표본 3,900세포라 작은 유형(pDC 등)은 지표가 흔들린다. 시드 하나, 단일 데이터셋이다.